# 4단계: 카탈로그 영문 선박명 → 한글 표기 (큰 LLM, vLLM)
런타임은 **가장 큰 GPU(G4 / H100 / A100 80GB)**로 바꾸세요. 첫 칸 설치 후 `import vllm`에서 오류가 나면 **런타임 → 세션 다시 시작** 후 두 번째 칸부터 실행하세요.
결과는 드라이브 `catalog_ko/out/`에 1만 개씩 저장됩니다(중간에 끊겨도 이어서 실행 가능).

In [1]:
!pip -q install vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, pandas as pd, torch
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/catalog_ko"
os.makedirs(f"{D}/out", exist_ok=True)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), round(mem), "GB")
MODEL = "Qwen/Qwen2.5-72B-Instruct-AWQ" if mem > 70 else "Qwen/Qwen2.5-32B-Instruct-AWQ"
print("model:", MODEL)


Mounted at /content/drive
NVIDIA RTX PRO 6000 Blackwell Server Edition 95 GB
model: Qwen/Qwen2.5-72B-Instruct-AWQ


In [3]:
from vllm import LLM, SamplingParams
llm = LLM(MODEL, max_model_len=4096, gpu_memory_utilization=0.92, enable_prefix_caching=True)
tok = llm.get_tokenizer()
SYSTEM = ("You convert ship names from an international vessel catalog into the Korean Hangul spelling that Korean "
          "port arrival records use (Korean loanword pronunciation, 외래어 표기). Keep numbers as digits. "
          "Abbreviations are read letter by letter (MSC -> 엠에스씨). Answer with the Hangul name only.")
shots = json.load(open(f"{D}/fewshot_pairs.json"))
def prompt(name):
    m = [{"role": "system", "content": SYSTEM}]
    for en, ko in shots:
        m += [{"role": "user", "content": en}, {"role": "assistant", "content": ko}]
    m.append({"role": "user", "content": str(name)})
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
greedy = SamplingParams(temperature=0, max_tokens=32)
sample = SamplingParams(temperature=0.8, top_p=0.95, n=3, max_tokens=32, seed=0)
clean = lambda t: t.strip().split("\n")[0].strip(" \"'`.")
for n in ["Glovis Supreme", "Pan Bonita", "Heung-A Sarah", "MSC Oscar", "Dongjin Fortune"]:
    print(n, "->", clean(llm.generate([prompt(n)], greedy, use_tqdm=False)[0].outputs[0].text))


(MainProcess pid=2393) INFO 10-31 10:05:11 [api_utils.py:286] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': True, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-72B-Instruct-AWQ'}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

(MainProcess pid=2393) INFO 10-31 10:05:30 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=2393) INFO 10-31 10:05:30 [model.py:2152] Using max model len 4096
(MainProcess pid=2393) INFO 10-31 10:05:31 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=16384.


model.safetensors.index.json:   0%|          | 0.00/179k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/11 [00:00<?, ?it/s]

(MainProcess pid=2393) INFO 10-31 10:05:32 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

(MainProcess pid=2393) WARNING 10-31 10:05:33 [system_utils.py:153] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=2393) INFO 10-31 10:11:31 [hf.py:648] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
Glovis Supreme -> 글로비스 슈프림
Pan Bonita -> 팬 보니타
Heung-A Sarah -> 흥아 사라
MSC Oscar -> 엠에스씨 오스카
Dongjin Fortune -> 동진포춘


In [4]:
cat = pd.read_csv(f"{D}/vessel_catalog.csv", encoding="utf-8-sig")
names = sorted(cat.vessel_name.astype(str).unique())
print(len(names), "unique names")
CH = 10000
for s in range(0, len(names), CH):
    fn = f"{D}/out/part_{s // CH:02d}.csv"
    if os.path.exists(fn):
        continue
    t = time.time()
    chunk = names[s:s + CH]
    ps = [prompt(n) for n in chunk]
    g = llm.generate(ps, greedy)
    r = llm.generate(ps, sample)
    rows = [[n, clean(a.outputs[0].text)] + [clean(o.text) for o in b.outputs] for n, a, b in zip(chunk, g, r)]
    pd.DataFrame(rows, columns=["vessel_name", "ko0", "ko1", "ko2", "ko3"]).to_csv(fn, index=False, encoding="utf-8-sig")
    print(fn, f"{time.time() - t:.0f}s")
print("done")


78594 unique names


Rendering prompts:   0%|          | 0/10000 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 10000/10000 [02:19<00:00, 71.57it/s, est. speed input: 59535.38 toks/s, output: 507.06 toks/s] 


Rendering prompts:   0%|          | 0/10000 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 30000/30000 [06:51<00:00, 72.92it/s, est. speed input: 60657.55 toks/s, output: 529.34 toks/s]


...

Rendering prompts:   0%|          | 0/8594 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 8594/8594 [01:58<00:00, 72.74it/s, est. speed input: 60578.86 toks/s, output: 470.81 toks/s] 


Rendering prompts:   0%|          | 0/8594 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 25782/25782 [05:19<00:00, 80.79it/s, est. speed input: 67287.01 toks/s, output: 554.64 toks/s] 


/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/catalog_ko/out/part_07.csv 457s
done
